
# V34 LIVE — Conditional Ranking Specialists

**Protected Kaggle champion: V14 = 0.586**

V34 does not train another generic 7-label model.

## Core idea

In the LIVE labeled data, every positive secondary attribute occurs inside `is_toxic=1`.
So the secondary problem is naturally conditional:

> **Given that a comment is toxic, which toxicity attributes does it contain?**

V34 trains two specialists only on toxic comments.

### Specialist A — semantic
Backbone: `FacebookAI/xlm-roberta-base`

Targets:
- `implicit_toxicity`
- `sarcasm`

### Specialist B — explicit / abuse-domain
Backbone: `saroarj/BanglaHateBert`

Targets:
- `contains_profanity`
- `threatening`
- `identity_attack`
- `sexual_content`

Both use:

- LIVE 5-fold grouped OOF
- positive-aware sampler
- mild positive BCE weighting
- **pairwise ranking loss**
- snapshot averaging
- no raw threshold dependence

At inference, conditional attribute scores are combined with the proven V14 toxicity score:

`attribute_rank = conditional_logit + α × toxicity_logit`

The final selector tests:
- fixed-count reranking (same number of positives as V14)
- tightly controlled prevalence changes

A candidate must improve across validation folds, not only in pooled validation F1.

**Exact Kaggle LIVE IDs only.**


In [ ]:

# 1. Upload V34 LIVE data
from google.colab import files
import pathlib,zipfile

uploaded=files.upload()
assert "v34_LIVE_data.zip" in uploaded

DATA_DIR=pathlib.Path("/content/v34_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v34_LIVE_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ",p.name,f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install + specialist checkpoint preflight
import sys,subprocess

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.57.6","tokenizers==0.22.1","sentencepiece==0.2.1",
    "scikit-learn","scipy","pandas","numpy","tqdm","safetensors"
])

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "git+https://github.com/csebuetnlp/normalizer"
])

from transformers import AutoConfig,AutoTokenizer

CHECKPOINTS={
    "semantic":"FacebookAI/xlm-roberta-base",
    "explicit":"saroarj/BanglaHateBert",
}

for k,v in CHECKPOINTS.items():
    print("Checking",k,v)
    _=AutoConfig.from_pretrained(v)
    _=AutoTokenizer.from_pretrained(v)

print("✅ V34 checkpoints available")


In [ ]:

# 3. Imports, strict LIVE verification, text preprocessing
import gc,re,math,json,random,unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,AutoModel,get_linear_schedule_with_warmup
)
from normalizer import normalize

SEED=2026
N_FOLDS=5
EPOCHS=3
SNAPSHOT_EPOCHS=[2,3]
MAX_LENGTH=128
BATCH=8
ACCUM=2
EVAL_BATCH=32
BACKBONE_LR=1.2e-5
HEAD_LR=3e-5
WEIGHT_DECAY=.01
WARMUP_RATIO=.10
RANK_LAMBDA=.18

LABELS=[
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]
LI={l:i for i,l in enumerate(LABELS)}

GROUPS={
    "semantic":["implicit_toxicity","sarcasm"],
    "explicit":["contains_profanity","threatening","identity_attack","sexual_content"],
}

CHECKPOINTS={
    "semantic":"FacebookAI/xlm-roberta-base",
    "explicit":"saroarj/BanglaHateBert",
}

RUN_DIR=Path("/content/v34_output")
RUN_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

seed_all(SEED)
assert torch.cuda.is_available(),"Enable T4 GPU"

train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
champ=pd.read_csv(DATA_DIR/"submission_v14.csv",dtype={"ID":str})
v2v=pd.read_csv(DATA_DIR/"v2_valid.csv",dtype={"ID":str})
v2t=pd.read_csv(DATA_DIR/"v2_test.csv",dtype={"ID":str})

assert len(train)==3297 and len(valid)==500 and len(test)==1000
assert test.ID.tolist()==champ.ID.tolist()
assert test.ID.head(4).tolist()==[
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]

def raw_clean(x):
    x=unicodedata.normalize("NFKC",str(x))
    x=x.replace("\u200c"," ").replace("\u200d"," ")
    return re.sub(r"\s+"," ",x).strip()

def bn_clean(x):
    x=raw_clean(x)
    try:
        x=normalize(x)
    except Exception:
        pass
    return re.sub(r"\s+"," ",x).strip()

def group_key(x):
    return raw_clean(x).lower()

for df in [train,valid,test]:
    df["raw_text"]=df.text.map(raw_clean)
    df["bn_text"]=df.text.map(bn_clean)
    df["group_key"]=df.text.map(group_key)

combined=pd.concat([train,valid],ignore_index=True)

print("✅ exact LIVE Kaggle partition verified")
display(pd.DataFrame({
    "train":train[LABELS].mean(),
    "validation":valid[LABELS].mean()
}).round(5))


In [ ]:

# 4. Recreate LIVE grouped 5-fold OOF split
groups=combined.groupby("group_key",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

gp=np.array([combined[l].sum() for l in LABELS],float)
total=float(len(combined))
rarity=1/np.sqrt(np.maximum(gp,1))
mat=groups[[f"sum_{l}" for l in LABELS]].to_numpy(float)

rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+.005*groups["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(groups))*1e-8))

fp=np.zeros((5,7))
fs=np.zeros(5)
assign={}

for idx in order:
    row=groups.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in LABELS],float)
    size=float(row["size"])

    costs=[]
    for f in range(5):
        p=fp.copy(); s=fs.copy()
        p[f]+=vec; s[f]+=size
        costs.append(
            np.mean((p/np.maximum(gp[None,:],1)-.2)**2)
            +.30*np.mean((s/total-.2)**2)
        )

    best=int(np.argmin(costs))
    assign[row.group_key]=best
    fp[best]+=vec
    fs[best]+=size

combined["fold"]=combined.group_key.map(assign).astype(int)

assert combined.groupby("group_key").fold.nunique().max()==1

print("Combined fold sizes:",combined.groupby("fold").size().to_dict())
print("Toxic-only target prevalence:")
display(
    combined[combined.is_toxic==1][
        ["implicit_toxicity","sarcasm","contains_profanity",
         "threatening","identity_attack","sexual_content"]
    ].mean().round(4)
)


In [ ]:

# 5. Conditional dataset / specialist model / ranking loss
class ConditionalDS(Dataset):
    def __init__(self,df,tok,text_col,targets,labels=True):
        self.x=df[text_col].astype(str).tolist()
        self.targets=targets
        self.y=df[targets].to_numpy(np.float32) if labels else None
        self.tok=tok

    def __len__(self):
        return len(self.x)

    def __getitem__(self,i):
        e=self.tok(
            self.x[i],truncation=True,max_length=MAX_LENGTH,padding=False
        )
        if self.y is not None:
            e["labels"]=self.y[i]
        return e

class Coll:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )

        b=self.tok.pad(
            items,padding=True,pad_to_multiple_of=8,return_tensors="pt"
        )
        if y is not None:
            b["labels"]=y
        return b

class Specialist(nn.Module):
    def __init__(self,name,nout):
        super().__init__()
        self.base=AutoModel.from_pretrained(name)
        h=self.base.config.hidden_size

        self.head=nn.Sequential(
            nn.Dropout(.12),
            nn.Linear(h*2,h),
            nn.GELU(),
            nn.Dropout(.18),
            nn.Linear(h,nout),
        )

    def forward(self,input_ids,attention_mask,token_type_ids=None):
        kw={"input_ids":input_ids,"attention_mask":attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"]=token_type_ids

        out=self.base(**kw)
        hs=out.last_hidden_state

        cls=hs[:,0]
        m=attention_mask.unsqueeze(-1).float()
        mean=(hs*m).sum(1)/m.sum(1).clamp_min(1.)

        return self.head(torch.cat([cls,mean],dim=1))

def pairwise_rank_loss(logits,y):
    losses=[]

    for j in range(logits.shape[1]):
        pos=logits[y[:,j]>.5,j]
        neg=logits[y[:,j]<.5,j]

        if len(pos)==0 or len(neg)==0:
            continue

        # hardest positives / negatives in the batch
        npair=min(8,len(pos),len(neg))

        pos_h=torch.sort(pos).values[:npair]
        neg_h=torch.sort(neg,descending=True).values[:npair]

        diff=pos_h[:,None]-neg_h[None,:]
        losses.append(
            torch.nn.functional.softplus(-diff).mean()
        )

    if not losses:
        return logits.sum()*0.0

    return torch.stack(losses).mean()

def specialist_loss(logits,y,pos_weight):
    bce=torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(),
        y,
        pos_weight=pos_weight,
    )

    rank=pairwise_rank_loss(logits.float(),y)

    return bce+RANK_LAMBDA*rank

@torch.no_grad()
def predict(model,loader):
    model.eval()
    parts=[]

    for b in loader:
        b.pop("labels",None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp)

        parts.append(torch.sigmoid(logits.float()).cpu().numpy())

    return np.concatenate(parts)


In [ ]:

# 6. Train one conditional specialist across 5 folds
fold_seeds=[2026,17,3407,1337,42]

def train_specialist(kind):
    targets=GROUPS[kind]
    name=CHECKPOINTS[kind]
    text_col="raw_text" if kind=="semantic" else "bn_text"

    oof=np.zeros((len(combined),len(targets)),np.float32)
    test_parts=[]
    history=[]

    tok=AutoTokenizer.from_pretrained(name)

    for fold in range(5):
        # TRAIN ONLY on toxic comments outside held fold
        tr=combined[
            (combined.fold!=fold)
            &(combined.is_toxic==1)
        ].copy()

        # evaluate on ALL held rows; conditional score will later be fused with P(toxic)
        va=combined[combined.fold==fold].copy()

        cache=RUN_DIR/f"{kind}_fold_{fold}.npz"
        hp=RUN_DIR/f"{kind}_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["idx"]; pv=z["valid"]; pt=z["test"]

            if np.array_equal(idx,va.index.to_numpy()):
                print("✅ reuse",kind,fold)
                oof[idx]=pv
                test_parts.append(pt)
                continue

        print(
            f"{kind} fold {fold} | toxic train={len(tr)} "
            f"| held all={len(va)}"
        )

        seed_all(fold_seeds[fold])
        gc.collect()
        torch.cuda.empty_cache()

        # mild oversampling of positive labels
        yy=tr[targets].to_numpy(float)

        if kind=="semantic":
            weights=1.0+2.0*yy[:,0]+1.0*yy[:,1]
        else:
            weights=(
                1.0
                +1.0*yy[:,0]
                +3.0*yy[:,1]
                +0.25*yy[:,2]
                +2.5*yy[:,3]
            )

        sampler=WeightedRandomSampler(
            torch.tensor(weights,dtype=torch.double),
            num_samples=len(tr),
            replacement=True,
        )

        # Mild positive BCE boost based on CONDITIONAL prevalence.
        pos=tr[targets].sum().to_numpy(float)
        neg=len(tr)-pos
        pw=np.sqrt(neg/np.maximum(pos,1))
        pw=np.clip(pw,1.0,2.5)

        pos_weight=torch.tensor(
            pw,dtype=torch.float32,device="cuda"
        )

        col=Coll(tok)

        tl=DataLoader(
            ConditionalDS(tr,tok,text_col,targets,True),
            batch_size=BATCH,
            sampler=sampler,
            num_workers=0,pin_memory=True,
            collate_fn=col
        )
        vl=DataLoader(
            ConditionalDS(va,tok,text_col,targets,False),
            batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,
            collate_fn=col
        )
        tel=DataLoader(
            ConditionalDS(test,tok,text_col,targets,False),
            batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,
            collate_fn=col
        )

        model=Specialist(name,len(targets)).to("cuda")

        backbone=[]
        head=[]

        for n,p in model.named_parameters():
            if n.startswith("base."):
                backbone.append(p)
            else:
                head.append(p)

        opt=torch.optim.AdamW([
            {"params":backbone,"lr":BACKBONE_LR,"weight_decay":WEIGHT_DECAY},
            {"params":head,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
        ])

        steps=math.ceil(len(tl)/ACCUM)*EPOCHS
        sched=get_linear_schedule_with_warmup(
            opt,int(steps*WARMUP_RATIO),steps
        )
        scaler=torch.amp.GradScaler("cuda")

        sv=[]; st=[]; fh=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            opt.zero_grad(set_to_none=True)

            run=0.; seen=0

            for step,b in enumerate(
                tqdm(tl,desc=f"{kind} f{fold} e{epoch}/{EPOCHS}")
            ):
                y=b.pop("labels").to("cuda",non_blocking=True)
                inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inp)

                loss=specialist_loss(logits,y,pos_weight)

                if not torch.isfinite(loss):
                    raise FloatingPointError("non-finite specialist loss")

                scaler.scale(loss/ACCUM).backward()

                run+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%ACCUM==0 or step+1==len(tl):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.)

                    old=scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()

                    if scaler.get_scale()>=old:
                        sched.step()

                    opt.zero_grad(set_to_none=True)

            rec={
                "kind":kind,"fold":fold,"epoch":epoch,
                "loss":run/max(seen,1)
            }
            print(rec)
            fh.append(rec); history.append(rec)
            pd.DataFrame(fh).to_csv(hp,index=False)

            if epoch in SNAPSHOT_EPOCHS:
                sv.append(predict(model,vl))
                st.append(predict(model,tel))

        pv=np.mean(sv,axis=0)
        pt=np.mean(st,axis=0)

        oof[va.index.to_numpy()]=pv
        test_parts.append(pt)

        np.savez_compressed(
            cache,
            idx=va.index.to_numpy(),
            valid=pv,
            test=pt,
        )

        del model,opt,sched,scaler,tl,vl,tel,sampler
        gc.collect()
        torch.cuda.empty_cache()

    ptest=np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/f"{kind}_oof_test.npz",
        oof=oof,test=ptest
    )

    pd.DataFrame(history).to_csv(
        RUN_DIR/f"{kind}_training_history.csv",index=False
    )

    return oof,ptest

semantic_oof,semantic_test=train_specialist("semantic")
explicit_oof,explicit_test=train_specialist("explicit")

print("✅ both V34 conditional specialists complete")


In [ ]:

# 7. Load V14 / V32 / V33 probability signals
v32=np.load(DATA_DIR/"v32_new_probabilities.npz")
v33=np.load(DATA_DIR/"v33_new_probabilities.npz")
v15=np.load(DATA_DIR/"v15_specialist.npz")
oldb=np.load(DATA_DIR/"old_bangla_snapshots.npz")
oldx=np.load(DATA_DIR/"old_xlmr_snapshots.npz")

NTR=len(train)
VIDX=np.arange(NTR,NTR+len(valid))

V15_LABELS=[
    "implicit_toxicity","contains_profanity",
    "threatening","sexual_content"
]
V15I={l:i for i,l in enumerate(V15_LABELS)}

def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

# Exact V14 validation/test probabilities
v14v={}
v14t={}

v14v["is_toxic"]=(
    .875*v2v.is_toxic_prob.to_numpy(float)
    +.125*oldx["valid"][1:3,:,0].mean(0)
)
v14t["is_toxic"]=(
    .875*v2t.is_toxic_prob.to_numpy(float)
    +.125*oldx["test"][1:3,:,0].mean(0)
)

j=LI["implicit_toxicity"]
v14v["implicit_toxicity"]=sigmoid(
    .9*logit(v2v.implicit_toxicity_prob)
    +.1*logit(oldb["valid"].mean(0)[:,j])
)
v14t["implicit_toxicity"]=sigmoid(
    .9*logit(v2t.implicit_toxicity_prob)
    +.1*logit(oldb["test"].mean(0)[:,j])
)

v14v["sarcasm"]=v2v.sarcasm_prob.to_numpy(float)
v14t["sarcasm"]=v2t.sarcasm_prob.to_numpy(float)

j=LI["contains_profanity"]
v14v["contains_profanity"]=(
    .95*v2v.contains_profanity_prob.to_numpy(float)
    +.05*oldb["valid"][0,:,j]
)
v14t["contains_profanity"]=(
    .95*v2t.contains_profanity_prob.to_numpy(float)
    +.05*oldb["test"][0,:,j]
)

j=LI["threatening"]
v14v["threatening"]=(
    .95*v2v.threatening_prob.to_numpy(float)
    +.05*oldx["valid"][0,:,j]
)
v14t["threatening"]=(
    .95*v2t.threatening_prob.to_numpy(float)
    +.05*oldx["test"][0,:,j]
)

j=LI["identity_attack"]
v14v["identity_attack"]=(
    .70*v2v.identity_attack_prob.to_numpy(float)
    +.30*oldb["valid"][2,:,j]
)
v14t["identity_attack"]=(
    .70*v2t.identity_attack_prob.to_numpy(float)
    +.30*oldb["test"][2,:,j]
)

v14v["sexual_content"]=v2v.sexual_content_prob.to_numpy(float)
v14t["sexual_content"]=v2t.sexual_content_prob.to_numpy(float)

print("✅ existing LIVE probability signals loaded")


In [ ]:

# 8. Metric + fold-stability helpers
yva=valid[LABELS].to_numpy(int)
valid_folds=combined.iloc[VIDX].fold.to_numpy(int)

v14_thresholds={
    "is_toxic":0.6850110311061144,
    "implicit_toxicity":0.1416,
    "sarcasm":0.175,
    "contains_profanity":0.245,
    "threatening":0.16558668018085876,
    "identity_attack":0.3814,
    "sexual_content":0.245,
}

v14_pred=np.column_stack([
    (v14v[l]>=v14_thresholds[l]).astype(int)
    for l in LABELS
])

def f1_pos(y,p):
    y=np.asarray(y,int); p=np.asarray(p,int)
    tp=((y==1)&(p==1)).sum()
    fp=((y==0)&(p==1)).sum()
    fn=((y==1)&(p==0)).sum()
    d=2*tp+fp+fn
    return float(2*tp/d) if d else 0.

def f1_macro(y,p):
    return float(np.mean([
        f1_pos((y==c).astype(int),(p==c).astype(int))
        for c in [0,1]
    ]))

def metric(label,y,p):
    return f1_macro(y,p) if label=="is_toxic" else f1_pos(y,p)

def topk(prob,k):
    k=int(max(0,min(k,len(prob))))
    out=np.zeros(len(prob),int)
    if k:
        out[np.argsort(-prob)[:k]]=1
    return out

def fold_scores_same_count(label,prob):
    j=LI[label]
    scores=[]
    base=[]

    for f in range(5):
        m=valid_folds==f

        # preserve V14 positive count INSIDE the same held fold
        k=int(v14_pred[m,j].sum())

        cp=topk(prob[m],k)
        bp=v14_pred[m,j]

        scores.append(metric(label,yva[m,j],cp))
        base.append(metric(label,yva[m,j],bp))

    return np.asarray(scores),np.asarray(base)


In [ ]:

# 9. Build conditional joint-ranking candidates
SEM_I={"implicit_toxicity":0,"sarcasm":1}
EXP_I={
    "contains_profanity":0,
    "threatening":1,
    "identity_attack":2,
    "sexual_content":3,
}

def source_probs(label,split):
    if split=="validation":
        tox={
            "v14tox":v14v["is_toxic"],
            "v32tox":v32["meta_validation"][:,0],
            "v33tox":v33["meta_validation"][:,0],
        }

        sources={"v14":v14v[label]}

        if label in SEM_I:
            sources["conditional_new"]=semantic_oof[VIDX,SEM_I[label]]

        if label in EXP_I:
            sources["conditional_new"]=explicit_oof[VIDX,EXP_I[label]]

        if label=="contains_profanity":
            sources["v32meta"]=v32["meta_validation"][:,LI[label]]
            sources["v33meta"]=v33["meta_validation"][:,LI[label]]

        if label=="identity_attack":
            sources["v32meta"]=v32["meta_validation"][:,LI[label]]
            sources["v33meta"]=v33["meta_validation"][:,LI[label]]

        if label in V15I:
            sources["v15"]=v15["oof"][VIDX,V15I[label]]

    else:
        tox={
            "v14tox":v14t["is_toxic"],
            "v32tox":v32["meta_test"][:,0],
            "v33tox":v33["meta_test"][:,0],
        }

        sources={"v14":v14t[label]}

        if label in SEM_I:
            sources["conditional_new"]=semantic_test[:,SEM_I[label]]

        if label in EXP_I:
            sources["conditional_new"]=explicit_test[:,EXP_I[label]]

        if label=="contains_profanity":
            sources["v32meta"]=v32["meta_test"][:,LI[label]]
            sources["v33meta"]=v33["meta_test"][:,LI[label]]

        if label=="identity_attack":
            sources["v32meta"]=v32["meta_test"][:,LI[label]]
            sources["v33meta"]=v33["meta_test"][:,LI[label]]

        if label in V15I:
            sources["v15"]=v15["test"][:,V15I[label]]

    return sources,tox

ALPHAS=[0.0,.10,.20,.35,.50,.75,1.0]

rows=[]
selected={}

for label in LABELS[1:]:
    j=LI[label]

    vsources,vtox=source_probs(label,"validation")
    tsources,ttox=source_probs(label,"test")

    base_global=metric(label,yva[:,j],v14_pred[:,j])
    base_test=champ[label].to_numpy(int)

    best={
        "label":label,
        "source":"v14",
        "tox":"none",
        "alpha":0.0,
        "validation_score":base_global,
        "gain":0.0,
        "wins":0,
        "ties":5,
        "losses":0,
        "test_changes":0,
        "validation_pred":v14_pred[:,j].copy(),
        "test_pred":base_test.copy(),
        "passed":False,
    }

    # A) FIXED-COUNT reranking — safest.
    k_global=int(v14_pred[:,j].sum())
    k_test=int(base_test.sum())

    for sname,sp in vsources.items():
        for tname,tp in vtox.items():
            for alpha in ALPHAS:
                score_v=logit(sp)+alpha*logit(tp)
                score_t=logit(tsources[sname])+alpha*logit(ttox[tname])

                vp=topk(score_v,k_global)
                pred_score=metric(label,yva[:,j],vp)

                fs,bs=fold_scores_same_count(label,score_v)
                wins=int((fs>bs+1e-12).sum())
                losses=int((fs<bs-1e-12).sum())
                ties=5-wins-losses

                tpred=topk(score_t,k_test)
                changes=int((tpred!=base_test).sum())

                gain=pred_score-base_global

                passed=(
                    gain>=.015
                    and wins>=3
                    and losses<=1
                )

                rec={
                    "label":label,
                    "policy":"fixed_count",
                    "source":sname,
                    "tox":tname,
                    "alpha":alpha,
                    "validation_score":pred_score,
                    "gain":gain,
                    "wins":wins,
                    "ties":ties,
                    "losses":losses,
                    "validation_positive_count":k_global,
                    "test_positive_count":k_test,
                    "test_changes":changes,
                    "passed":passed,
                }
                rows.append(rec)

                key=(
                    passed,
                    pred_score,
                    wins,
                    -losses,
                    -changes,
                )

                current_key=(
                    best["passed"],
                    best["validation_score"],
                    best["wins"],
                    -best["losses"],
                    -best["test_changes"],
                )

                if key>current_key:
                    best={
                        **rec,
                        "validation_pred":vp,
                        "test_pred":tpred,
                    }

    # B) controlled prior-rate change — only for a large, stable improvement.
    true_train=float(train[label].mean())

    for sname,sp in vsources.items():
        for tname,tp in vtox.items():
            for alpha in [.0,.25,.50,.75]:
                joint_v=logit(sp)+alpha*logit(tp)
                joint_t=logit(tsources[sname])+alpha*logit(ttox[tname])

                for scale in [.80,1.00,1.20]:
                    rate=true_train*scale
                    vk=int(round(rate*len(valid)))
                    tk=int(round(rate*len(test)))

                    vp=topk(joint_v,vk)
                    tpred=topk(joint_t,tk)

                    pred_score=metric(label,yva[:,j],vp)
                    gain=pred_score-base_global

                    # Fold wins against V14 using approximately the same prior rate.
                    fs=[]; bs=[]
                    for f in range(5):
                        m=valid_folds==f
                        kk=int(round(rate*m.sum()))
                        cp=topk(joint_v[m],kk)

                        fs.append(metric(label,yva[m,j],cp))
                        bs.append(metric(label,yva[m,j],v14_pred[m,j]))

                    fs=np.asarray(fs); bs=np.asarray(bs)
                    wins=int((fs>bs+1e-12).sum())
                    losses=int((fs<bs-1e-12).sum())
                    ties=5-wins-losses

                    changes=int((tpred!=base_test).sum())

                    passed=(
                        gain>=.040
                        and wins>=3
                        and losses<=1
                    )

                    rec={
                        "label":label,
                        "policy":"prior_rate",
                        "source":sname,
                        "tox":tname,
                        "alpha":alpha,
                        "scale":scale,
                        "validation_score":pred_score,
                        "gain":gain,
                        "wins":wins,
                        "ties":ties,
                        "losses":losses,
                        "validation_positive_count":vk,
                        "test_positive_count":tk,
                        "test_changes":changes,
                        "passed":passed,
                    }
                    rows.append(rec)

                    key=(
                        passed,
                        pred_score,
                        wins,
                        -losses,
                        -changes,
                    )
                    current_key=(
                        best["passed"],
                        best["validation_score"],
                        best["wins"],
                        -best["losses"],
                        -best["test_changes"],
                    )

                    if key>current_key:
                        best={
                            **rec,
                            "validation_pred":vp,
                            "test_pred":tpred,
                        }

    selected[label]=best

search=pd.DataFrame(rows)
search.to_csv(
    RUN_DIR/"v34_candidate_search.csv",index=False
)

print("Selected attribute candidates:")
display(pd.DataFrame([
    {k:v for k,v in selected[l].items()
     if k not in ["validation_pred","test_pred"]}
    for l in LABELS[1:]
]).round(5))


In [ ]:

# 10. Build protected V34 submission
val_pred=v14_pred.copy()
test_pred=champ[LABELS].to_numpy(int).copy()

for label in LABELS[1:]:
    j=LI[label]
    if selected[label]["passed"]:
        val_pred[:,j]=selected[label]["validation_pred"]
        test_pred[:,j]=selected[label]["test_pred"]

def official(y,p):
    ss=[
        metric(l,y[:,j],p[:,j])
        for j,l in enumerate(LABELS)
    ]
    return float(.5*ss[0]+.5*np.mean(ss[1:])),ss

vscore,scores=official(yva,val_pred)

report=pd.DataFrame({
    "label":LABELS,
    "validation_score":scores,
    "v14_validation_score":[
        metric(l,yva[:,j],v14_pred[:,j])
        for j,l in enumerate(LABELS)
    ],
    "selected":[
        False if l=="is_toxic" else bool(selected[l]["passed"])
        for l in LABELS
    ],
    "test_changes_vs_v14":[
        int((test_pred[:,j]!=champ[l].to_numpy(int)).sum())
        for j,l in enumerate(LABELS)
    ],
    "test_positive_rate":test_pred.mean(0),
})
report.to_csv(
    RUN_DIR/"v34_final_validation_report.csv",index=False
)

sub=champ.copy()
for j,l in enumerate(LABELS):
    sub[l]=test_pred[:,j]

assert sub.ID.tolist()==test.ID.tolist()
assert sub.ID.tolist()==champ.ID.tolist()
assert sub.ID.head(4).tolist()==[
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]
assert sub[LABELS].isin([0,1]).all().all()

sub.to_csv(
    RUN_DIR/"submission_v34_LIVE.csv",index=False
)

summary={
    "version":"v34_live_conditional_ranking_specialists",
    "protected_public_lb":0.586,
    "validation_official_score":float(vscore),
    "selected_labels":[
        l for l in LABELS[1:]
        if selected[l]["passed"]
    ],
    "total_test_changes":int(
        (test_pred!=champ[LABELS].to_numpy(int)).sum()
    ),
    "SUBMIT_CANDIDATE":bool(
        vscore>=.645
        and len([
            l for l in LABELS[1:]
            if selected[l]["passed"]
        ])>=1
    ),
}

(RUN_DIR/"v34_summary.json").write_text(
    json.dumps(summary,indent=2)
)

print(json.dumps(summary,indent=2))
display(report.round(5))

if summary["SUBMIT_CANDIDATE"]:
    print("\n✅ V34 passed the stricter live gate.")
else:
    print("\n❌ V34 did not pass. Send results ZIP; do not submit.")


In [ ]:

# 11. Package V34 results
np.savez_compressed(
    RUN_DIR/"v34_new_probabilities.npz",
    semantic_oof=semantic_oof,
    semantic_test=semantic_test,
    explicit_oof=explicit_oof,
    explicit_test=explicit_test,
)

zip_path=Path("/content/v34_results.zip")

with zipfile.ZipFile(
    zip_path,"w",compression=zipfile.ZIP_DEFLATED
) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
